# Homework: Обучение Multi-Branch MLP на размеченных данных

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

from model import MultiBranchMLP
from data_module import SemiSupervisedDataModule
from lightning_module import SemiSupervisedLightningModule

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


## 1. Загрузка данных


In [ ]:
data_dir = '../data'

dm = SemiSupervisedDataModule(
    data_dir=data_dir,
    batch_size=256,  
    num_workers=4,
    use_unlabeled=True  # Используем неразмеченные данные
)

dm.setup()

print(f'Input dimension: {dm.input_dim}')
print(f'Number of classes: {dm.n_classes}')
print(f'Labeled train samples: {len(dm.train_labeled_dataset)}')
print(f'Test samples: {len(dm.test_dataset)}')


## 2. Анализ дисбаланса классов и вычисление весов


In [ ]:
train_labels = dm.train_labeled_dataset.y

unique_labels = np.unique(train_labels)
class_weights = compute_class_weight(
    'balanced',
    classes=unique_labels,
    y=train_labels
)

print(f'Class weights: {dict(zip(unique_labels, class_weights))}')

class_weights_tensor = torch.FloatTensor(class_weights)


## 3. Создание модели


In [ ]:
model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=256, 
    output_dim=dm.n_classes,
    num_blocks=4,
    dropout=0.05, 
    combine_mode='concat'
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')


## 4. Создание Lightning модуля


In [ ]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

lightning_model = SemiSupervisedLightningModule(
    model=model,
    loss_fn=loss_fn,
    optimizer_type='adam',
    learning_rate=2e-3,
    task_type='multiclass',
    temperature=0.5
)


## 5. Обучение модели


In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints',
    filename='best_model-{epoch:02d}-{val_accuracy:.4f}',
    monitor='val_accuracy',
    mode='max',
    save_top_k=1,
    save_last=True
)

trainer = Trainer(
    max_epochs=100,
    callbacks=[checkpoint_callback],
    enable_checkpointing=True,
    logger=True,
    enable_progress_bar=True,
    enable_model_summary=True,
    accelerator='auto',
    devices='auto',
    check_val_every_n_epoch=2
)

trainer.fit(lightning_model, dm)


## 6. Оценка на тестовой выборке


In [ ]:
best_model_path = checkpoint_callback.best_model_path
print(f'Loading best model from: {best_model_path}')

if best_model_path:
    best_model = SemiSupervisedLightningModule.load_from_checkpoint(
        best_model_path,
        model=model,
        loss_fn=loss_fn,
        optimizer_type='adamw',
        learning_rate=1e-3,
        task_type='multiclass'
    )
else:
    best_model = lightning_model

test_results = trainer.test(best_model, dm)

print('\n=== Финальные результаты на тестовой выборке ===')
for key, value in test_results[0].items():
    print(f'{key}: {value:.4f}')
